In [15]:
# model.py
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 28 -> 14
        x = self.pool(F.relu(self.conv2(x)))  # 14 -> 7
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)

In [26]:
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_train_transform(aug_type):
    if aug_type == "none":
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])

    if aug_type == "rotation":
        return transforms.Compose([
            transforms.RandomAffine(degrees=10),
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])

    if aug_type == "translation":
        return transforms.Compose([
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])

    if aug_type == "rotation+translation":
        return transforms.Compose([
            transforms.RandomAffine(degrees=10, translate=(0.1, 0.1)),
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])


In [27]:
                                                                                                                                                                                                                                                                                        # train.py
import torch
from torch import nn, optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for aug_type in ["none", "rotation", "translation", "rotation+translation"]:
  set_seed(42)

  transform_train = get_train_transform(aug_type)

  train_data = datasets.MNIST("./data", train=True, download=True, transform=transform_train)
  train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

  model = SimpleCNN().to(device)
  criterion = nn.CrossEntropyLoss()
  optimizer = optim.Adam(model.parameters(), lr=0.001)

  for epoch in range(5):
      model.train()
      total_loss = 0

      for images, labels in train_loader:
          images, labels = images.to(device), labels.to(device)

          optimizer.zero_grad()
          outputs = model(images)
          loss = criterion(outputs, labels)
          loss.backward()
          optimizer.step()

          total_loss += loss.item()

      avg_loss = total_loss / len(train_loader)
      print(f"[{aug_type}] Epoch {epoch+1}/5 - Loss: {avg_loss:.4f}")

  # torch.save(model.state_dict(), "mnist_cnn.pth")
  # print("Model saved as mnist_cnn.pth")


  transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

  test_data = datasets.MNIST("./data", train=False, download=True, transform=transform_test)
  test_loader = DataLoader(test_data, batch_size=1000, shuffle=False)

  model.eval()

  correct = 0
  total = 0

  with torch.no_grad():
      for images, labels in test_loader:
          images, labels = images.to(device), labels.to(device)
          outputs = model(images)
          predictions = outputs.argmax(dim=1)

          correct += (predictions == labels).sum().item()
          total += labels.size(0)

  accuracy = 100 * correct / total
  print(f"[{aug_type}] Final Test Accuracy: {accuracy:.2f}%\n")

[none] Epoch 1/5 - Loss: 0.1629
[none] Epoch 2/5 - Loss: 0.0524
[none] Epoch 3/5 - Loss: 0.0384
[none] Epoch 4/5 - Loss: 0.0308
[none] Epoch 5/5 - Loss: 0.0243
[none] Final Test Accuracy: 99.12%

[rotation] Epoch 1/5 - Loss: 0.1862
[rotation] Epoch 2/5 - Loss: 0.0686
[rotation] Epoch 3/5 - Loss: 0.0523
[rotation] Epoch 4/5 - Loss: 0.0426
[rotation] Epoch 5/5 - Loss: 0.0380
[rotation] Final Test Accuracy: 99.20%

[translation] Epoch 1/5 - Loss: 0.2754
[translation] Epoch 2/5 - Loss: 0.0991
[translation] Epoch 3/5 - Loss: 0.0759
[translation] Epoch 4/5 - Loss: 0.0637
[translation] Epoch 5/5 - Loss: 0.0563
[translation] Final Test Accuracy: 99.12%

[rotation+translation] Epoch 1/5 - Loss: 0.2993
[rotation+translation] Epoch 2/5 - Loss: 0.1150
[rotation+translation] Epoch 3/5 - Loss: 0.0876
[rotation+translation] Epoch 4/5 - Loss: 0.0776
[rotation+translation] Epoch 5/5 - Loss: 0.0691
[rotation+translation] Final Test Accuracy: 99.28%

